In [1]:
import os
# Pinning versions to satisfy lunarnav (numpy < 2) and colab (pandas 2.2.3) requirements
!pip install "numpy>=1.24,<2.0" "pandas==2.2.3"
print("\n--- SUCCESS ---\nPlease go to 'Runtime' -> 'Restart session' now to apply changes.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 116.6 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 3.0.5
    Uninstalling pandas-3.0.5:
      Successfully uninstalled pandas-3.0.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.

--- SUCCESS ---
Please go to 'Runtime' -> 'Restart session' now to apply changes.


# lunar-navigation — honest evaluation (Run all)

**Before running:** Runtime → Change runtime type → **A100 GPU** (Colab Pro).

**Secrets** (key icon in left sidebar): `GITHUB_TOKEN`, `KAGGLE_KEY`. If `KAGGLE_KEY` is a raw API key (not full `kaggle.json`), also set `KAGGLE_USERNAME`.

Then **Run all** and close the laptop. Depth maps, training, baselines, and results push run here — not on your Mac.

In [2]:
import json
import os
import subprocess
import sys

from google.colab import userdata

GITHUB_TOKEN = userdata.get("lunar-nav-colab")
KAGGLE_SECRET = userdata.get("lunar-nav-data-token")

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
kaggle_path = os.path.expanduser("~/.kaggle/kaggle.json")
try:
    parsed = json.loads(KAGGLE_SECRET)
    with open(kaggle_path, "w") as handle:
        json.dump(parsed, handle)
except json.JSONDecodeError:
    username = userdata.get("KAGGLE_USERNAME")
    if not username:
        raise RuntimeError(
            "KAGGLE_KEY is not valid JSON. Add KAGGLE_USERNAME secret with your Kaggle username."
        )
    with open(kaggle_path, "w") as handle:
        json.dump({"username": username, "key": KAGGLE_SECRET}, handle)
os.chmod(kaggle_path, 0o600)

REPO_URL = f"https://{GITHUB_TOKEN}@github.com/AmeyaKI/lunar-navigation.git"
BRANCH = "fix/leakage-distill"
REPO_DIR = "/content/lunar-navigation"

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Runtime → Change runtime type → A100 GPU (Colab Pro), then Run all again."
    )
print(f"GPU: {torch.cuda.get_device_name(0)}")

GPU: NVIDIA A100-SXM4-40GB


In [3]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

CACHE_ROOT = Path("/content/drive/MyDrive/lunar-navigation-cache")
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
(CACHE_ROOT / "kagglehub").mkdir(exist_ok=True)
(CACHE_ROOT / "checkpoints").mkdir(exist_ok=True)
(CACHE_ROOT / "results").mkdir(exist_ok=True)
print(f"Cache root: {CACHE_ROOT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cache root: /content/drive/MyDrive/lunar-navigation-cache


In [4]:
from lunarnav.data import prepare_data

data = prepare_data(cache_root=CACHE_ROOT, kaggle_cache_dir=CACHE_ROOT / "kagglehub")
print(f"Frames after filter: {data['frame_count']}")
print(
    f"Train boxes: {len(data['train_df'])}, "
    f"Val: {len(data['val_df'])}, Test: {len(data['test_df'])}"
)

Using Colab cache for faster access to the 'artificial-lunar-rocky-landscape-dataset' dataset.


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/lunar-navigation-cache/dataset/images/render'

In [ ]:
import torch
from lunarnav.depth import generate_depth_maps

device = torch.device("cuda")
generated = generate_depth_maps(data["image_dir"], data["depth_dir"], device)
print(f"Newly generated depth maps: {generated}")

In [ ]:
from torch.utils.data import DataLoader

from lunarnav.data import RockDataset
from lunarnav.model import build_rgb_resnet18
from lunarnav.train import set_seed, train_model

set_seed(42)
train_ds = RockDataset(data["train_df"], data["image_dir"], data["depth_dir"])
val_ds = RockDataset(data["val_df"], data["image_dir"], data["depth_dir"])
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

device = torch.device("cuda")
model = build_rgb_resnet18(num_outputs=2).to(device)
ckpt_path = CACHE_ROOT / "checkpoints" / "best_resnet18.pt"
history = train_model(model, train_loader, val_loader, device, ckpt_path)
print(history)

In [ ]:
import platform

import torch
from lunarnav.eval import run_full_evaluation

hardware = {
    "gpu": torch.cuda.get_device_name(0),
    "cpu": platform.processor() or platform.machine(),
}
results = run_full_evaluation(
    data["train_df"],
    data["test_df"],
    data["image_dir"],
    data["depth_dir"],
    model,
    device,
    hardware,
    int(data["frame_count"]),
    results_dir=f"{REPO_DIR}/results",
)
print("Evaluation complete. See RESULTS.md and HANDOFF.md")

In [ ]:
import shutil
import subprocess
from pathlib import Path

repo = Path(REPO_DIR)
for name in ("RESULTS.md", "HANDOFF.md", "results"):
    src = repo / name
    dst = CACHE_ROOT / name
    if src.is_dir():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
    elif src.exists():
        shutil.copy2(src, dst)

os.chdir(REPO_DIR)
subprocess.run(["git", "config", "user.email", "colab@users.noreply.github.com"], check=False)
subprocess.run(["git", "config", "user.name", "colab-bot"], check=False)
subprocess.run(["git", "add", "RESULTS.md", "HANDOFF.md", "results/"], check=True)
subprocess.run(
    ["git", "commit", "-m", "colab: add evaluation results and HANDOFF.md"],
    check=False,
)
remote_url = f"https://{GITHUB_TOKEN}@github.com/AmeyaKI/lunar-navigation.git"
push_ok = subprocess.run(["git", "push", remote_url, "HEAD:fix/leakage-distill"]).returncode == 0

if not push_ok:
    print("GIT PUSH FAILED — copies saved to Drive; full file contents below:")
    for path in [repo / "RESULTS.md", repo / "HANDOFF.md"]:
        print(f"\n===== {path.name} =====\n")
        print(path.read_text())
else:
    print("Pushed RESULTS.md, HANDOFF.md, and results/ to fix/leakage-distill")